# Transformers
Para esta practica se usaran 3 notebooks:
1. Notebook 5.1 Scrap Comments PlayStore. Para obtener el dataset con comentarios a clasificar usando un LLM.
2. Notebook 5.2. Uso de LLMs metodo A.
3. Notebook 5.3. Uso de LLMs metodo B.

Teoria:
1. [Presentacion 5.1 La Revolucion de los Transformers](https://docs.google.com/presentation/d/1vg2svQej5ENBQoWF8WFCiu44TgMabgBKmg5qxQgxeX0/edit?usp=sharing)
2. [Video La Revolucion de los Transformers](/Workspace/Users/ronaldriveraab@gmail.com/UValle/5.1 Transformers y LLMs/La_Revolución_Transformer.mp4)
3. [Video Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)


## 5.1 Scrap Comments PlayStore
Obtener comentarios de Apps desde PlayStore (Google)
Scrip para obtener comentarios de aplicaciones.

En este caso se obtendra los comentarios dejados por clientes de distintas Apps de billeteras moviles de Bolivia.

In [0]:
%sql
use catalog `workspace`; select * from `default`.`googleplay_apps_reviews` limit 100;

id,date,rating,version,content,Billetera,Store
abd737e8-2ca7-44e4-85ae-b473f06fb027,2025-05-12,5.0,7.17.1,muy buena,Yape,PlayStore
8d896433-6c09-41f0-a6d5-bfffb02b25e8,2025-05-07,5.0,7.17.1,muy buena,Yape,PlayStore
75186e90-e7da-4e12-9fa9-98f2cd0856df,2025-10-27,5.0,7.17.1,buen servecio,Yape,PlayStore
bbd1cbc3-b300-40d8-af8c-f779eb361b80,2025-05-06,5.0,7.17.1,muy buena,Yape,PlayStore
5e5f9276-04ac-4ee6-9b4c-1141baf7dc62,2025-05-30,5.0,7.17.1,excelente aplicación,Yape,PlayStore
af483522-8516-451b-9327-403186458d00,2025-05-16,5.0,7.17.1,Muy buena,Yape,PlayStore
0f8c9ebd-d32a-41f6-bc67-2793952f8310,2025-05-29,5.0,7.17.1,exelente gracias,Yape,PlayStore
50ca89af-e8b4-4091-8060-ac323cc3d68c,2025-05-10,5.0,7.17.1,muy bueno,Yape,PlayStore
569e76f8-f71b-420c-b7f4-b1f0071abfd8,2025-05-08,5.0,7.17.1,genial gracias,Yape,PlayStore
6030b315-639e-4a6f-8002-21bbed8b929a,2025-05-24,5.0,7.17.1,muy bueno,Yape,PlayStore


## 0. Requirements

In [0]:
%pip install google_play_scraper

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

Rename some columns and filter others.

## 1. PlayStore scraping

In [0]:
import pandas as pd
from google_play_scraper import Sort, reviews_all, reviews, app
import numpy as np
import json, os, uuid

In [0]:
# ------------------- Yolo
yolo = reviews_all(
    'bo.com.yolopago',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, # defaults to 0
    sort=Sort.MOST_RELEVANT # defaults to Sort.MOST_RELEVANT
    #filter_score_with=5 # defaults to None(means all score)
)

# ------------------- Yape
yape = reviews_all(
    'com.bcp.bo.wallet',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 
   
)

# ------------------- Yasta
yasta = reviews_all(
    'com.busa.wallet',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 
)

# ------------------- Altoke
altoke= reviews_all(
    'com.bancosol.altoke',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 

)

# ------------------- TigoM
tigom= reviews_all(
    'bo.tigo.mfsapp',
    lang='es', 
    country='BO',
    sleep_milliseconds=500, 
    sort=Sort.MOST_RELEVANT 
)


In [0]:
# Convert to tables
yolo_tabla=pd.json_normalize(yolo)
yolo_tabla['Billetera']='Yolo'

yape_tabla=pd.json_normalize(yape)
yape_tabla['Billetera']='Yape'

tigom_tabla=pd.json_normalize(tigom)
tigom_tabla['Billetera']='TigoM'

yasta_tabla=pd.json_normalize(yasta)
yasta_tabla['Billetera']='Yasta'

altoke_tabla=pd.json_normalize(altoke)
altoke_tabla['Billetera']='Altoke'

In [0]:
# Concatenacion
playstore = pd.concat([yolo_tabla, yape_tabla, tigom_tabla, yasta_tabla, altoke_tabla], ignore_index=True)

## 3. Merge

In [0]:
# Filter columns
playstore=playstore[['reviewId','at','score','reviewCreatedVersion','content','Billetera']]
playstore['Store']='PlayStore'

# Rename columns
playstore = playstore.rename(
    columns={
        'reviewId': 'id',
        'at': 'date',
        'score': 'rating',
        'reviewCreatedVersion': 'version'
    }
)

playstore.head()

,id,date,rating,version,content,Billetera,Store
0,26ee1823-1bc7-4072-9e48-dd3b20fb41fd,2026-04-15 21:04:24,5,3.5.2,La aplicación Yolo Pago está muy bien cumple s...,Yolo,PlayStore
1,fc9f60b9-0502-4df0-ac24-c9fba967cb9b,2026-05-17 12:56:51,3,3.6.2,Es buena la app lo molesto es que a cada rato ...,Yolo,PlayStore
2,cea238d6-605a-4b7e-806e-22a5f1d3322f,2026-04-20 22:56:59,1,3.5.2,me hizo quedar mal con la ventera porque cuand...,Yolo,PlayStore
3,2b94432f-f0d1-4cf2-937d-8da13b91e934,2026-05-28 05:21:30,5,3.6.2,"Excelente app, deberia ser mas rapida en notif...",Yolo,PlayStore
4,b9b41f26-5b04-4d91-9082-ad6dc397c744,2026-05-21 22:41:35,2,3.6.2,todo el tiempo está en mantenimiento está app....,Yolo,PlayStore


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, FloatType

playstore_clean = playstore.copy()

schema = StructType([
    StructField("id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("rating", FloatType(), True),
    StructField("version", StringType(), True),
    StructField("content", StringType(), True),
    StructField("Billetera", StringType(), True),
    StructField("Store", StringType(), True)
])

#Stores_clean['date'] = pd.to_datetime(Stores_clean['date'], errors='coerce')
#Stores_clean['rating'] = pd.to_numeric(Stores_clean['rating'], errors='coerce')

df = spark.createDataFrame(playstore_clean, schema=schema)

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.googleplay_apps_reviews")